> 🔑 **Lösungsversion für Lehrkräfte.** Die Schülerversion im Hauptordner wird daraus mit
> `python3 material/quelle/schuelerversion_erstellen.py` erzeugt: Code zwischen `# >>> LOESUNG` und
> `# <<< LOESUNG` wird entfernt, Zeilen mit `# | ` werden zum Platzhalter für die Lernenden.

# 11 – Vollvernetztes Netz (FCNN) systematisch trainieren und verbessern

**Lernsituation „KI-Sichtprüfstation“ · Doppelstunde 5**

In Notebook 10 hat das Netz gelernt – aber oft mit einer deutlichen Lücke zwischen Trainings- und Validierungsgenauigkeit. Jetzt arbeiten wir mit dem **großen Datensatz**, bauen die Vorverarbeitung **ins Modell** ein und lernen Werkzeuge gegen das „Auswendiglernen“ kennen.

### Lernziele
Du kannst …
- Lernkurven deuten und Über- sowie Unteranpassung erkennen,
- Datenaugmentierung, Dropout und EarlyStopping begründet einsetzen,
- Hyperparameter-Experimente planen, durchführen und vergleichen,
- ein Modell mit Testdaten und Konfusionsmatrix bewerten und speichern.

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Grundlagen lesen / interaktiv erkunden |
| ✏️ | Aufgabe: Code **selbst schreiben** (Zellen mit `# TODO`) |
| 🔍 | Experiment / Beobachtung – Ergebnisse notieren |
| 🧪 | Teilzielkontrolle |
| 🏭 | Bezug zur industriellen Fertigung |
| 🍓 | Hinweis für den Raspberry Pi |

In [ ]:
# =============================================================================
# Vorbereitung: Hilfsmodul finden (klappt im Hauptordner und in loesungen/)
# =============================================================================
import sys
from pathlib import Path

ORDNER = Path.cwd() if (Path.cwd() / "ki_hilfen.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ORDNER))

import numpy as np
import cv2
import matplotlib.pyplot as plt
import ki_hilfen as kh

import time
import pandas as pd
import keras
from keras import layers
import tensorflow as tf

kh.setze_seed(42)
print("TensorFlow", tf.__version__)

## 1 · Einstellungen und Daten

Ab jetzt lädt `kh.lade_datensatz()` die Bilder. Die Funktion macht dasselbe wie dein `lade_bilder()` aus Notebook 10 – speichert das Ergebnis aber zusätzlich in einem Zwischenspeicher (`.cache/`). Beim zweiten Mal dauert das Laden nur noch Sekunden.

🍓 Der erste Ladevorgang des großen Datensatzes dauert auf dem Raspberry Pi etwa 1 – 2 Minuten.

In [ ]:
# =============================================================================
# Einstellungen (Hyperparameter) – hier wird experimentiert
# =============================================================================
DATENSATZ = ORDNER / "dataset"      # großer Datensatz (1735 Bilder) oder eigener
GROESSE = 32                        # Bilder werden auf GROESSE x GROESSE verkleinert
FARBE = "grau"                      # "grau" (1 Kanal) oder "rgb" (3 Kanäle)
BATCH = 32
EPOCHEN = 200                       # Obergrenze – EarlyStopping beendet meist früher
LERNRATE = 1e-3

In [ ]:
X, y, klassen = kh.lade_datensatz(DATENSATZ, GROESSE, FARBE)
X_train, X_val, X_test, y_train, y_val, y_test = kh.teile_datensatz(X, y, (0.70, 0.15, 0.15))
ANZAHL_KLASSEN = len(klassen)
EINGABE = X_train.shape[1:]

print("Klassen:", klassen)
print("Eingabeform:", EINGABE, "| Werte:", X.dtype, X.min(), "…", X.max())
print(f"Training {len(X_train)} · Validierung {len(X_val)} · Test {len(X_test)}")
kh.zeige_beispiele(X_train, y_train, klassen)

### 📖 Zwei Änderungen gegenüber Notebook 10

**1. Normierung im Modell.** Die Bilder bleiben im Speicher `uint8` (0 … 255). Die erste Schicht `layers.Rescaling(1/255)` teilt durch 255. So steckt die Vorverarbeitung *im* gespeicherten Modell – an der Prüfstation kann man sie nicht mehr vergessen. 🏭 *Ein typischer Fehler in der Praxis: Training mit normierten Bildern, Betrieb ohne Normierung – und niemand merkt, warum die Station plötzlich nur noch rät.*

**2. Labels als Zahlen.** Statt One-Hot-Vektoren verwenden wir die Verlustfunktion `sparse_categorical_crossentropy`. Sie rechnet genau dasselbe, erledigt die One-Hot-Umwandlung aber intern.

## 2 · Datenaugmentierung
Bei der **Augmentierung** wird jedes Trainingsbild in jeder Epoche zufällig leicht verändert (gedreht, verschoben, gezoomt …). Das Netz sieht so viel mehr Varianten und lernt weniger auswendig. Augmentiert werden **nur die Trainingsdaten**.

In [ ]:
augmentierung = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),             # ± 0,08 · 360° ≈ ± 29°
    layers.RandomTranslation(0.1, 0.1),      # ± 10 % verschieben
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.2),
], name="augmentierung")

# Ein Bild neunmal zufällig verändern
bild = X_train[:1].astype("float32")
plt.figure(figsize=(7, 7))
for i in range(9):
    aug = np.clip(augmentierung(bild, training=True)[0].numpy(), 0, 255).astype("uint8")
    plt.subplot(3, 3, i + 1)
    plt.imshow(aug[..., 0] if aug.shape[-1] == 1 else aug, cmap="gray", vmin=0, vmax=255)
    plt.axis("off")
plt.suptitle("Augmentierte Varianten eines Trainingsbildes"); plt.tight_layout(); plt.show()

🔍 **Diskutiert:** Ist ein *horizontales Spiegeln* für unsere Symbole sinnvoll? Und für ein Bauteil, bei dem die **Lage** geprüft werden soll (richtig herum oder verdreht)? 🏭 Augmentierung darf nie das verändern, was die Klasse ausmacht!

In [ ]:
train_ds = kh.als_tf_dataset(X_train, y_train, BATCH, mischen=True, augmentierung=augmentierung)
val_ds = kh.als_tf_dataset(X_val, y_val, BATCH)
test_ds = kh.als_tf_dataset(X_test, y_test, BATCH)
print("Stapel pro Epoche:", len(train_ds))

## 3 · Das Modell als Funktion

### ✏️ Aufgabe 1 – `baue_fcnn()` vervollständigen
Damit wir schnell verschiedene Netze vergleichen können, bauen wir das Modell in einer Funktion. Ergänze die Schleife: Für **jede** Zahl in `neuronen` soll eine `Dense`-Schicht mit ReLU entstehen, gefolgt von einer `Dropout`-Schicht (nur wenn `dropout > 0`).

📖 **Dropout** schaltet beim Training in jedem Schritt zufällig einen Anteil der Neuronen ab. Das Netz kann sich dadurch nicht auf einzelne Neuronen verlassen und verallgemeinert besser. Bei der Vorhersage ist Dropout automatisch aus.

In [ ]:
def baue_fcnn(neuronen=(256, 128, 64), dropout=0.3):
    modell = keras.Sequential(name="fcnn")
    modell.add(keras.Input(shape=EINGABE))
    modell.add(layers.Rescaling(1 / 255))
    modell.add(layers.Flatten())
    for n in neuronen:
        # >>> LOESUNG
        # | pass   # TODO: Dense-Schicht mit n Neuronen (ReLU) und ggf. Dropout ergänzen
        modell.add(layers.Dense(n, activation="relu"))
        if dropout > 0:
            modell.add(layers.Dropout(dropout))
        # <<< LOESUNG
    modell.add(layers.Dense(ANZAHL_KLASSEN, activation="softmax"))
    return modell


test_modell = baue_fcnn((32, 16), dropout=0.5)
test_modell.summary()
art = [type(s).__name__ for s in test_modell.layers]
kh.pruefe(lambda: art.count("Dense") == 3, "Anzahl Dense-Schichten stimmt",
          "Für neuronen=(32, 16) erwarten wir 2 versteckte + 1 Ausgabeschicht")
kh.pruefe(lambda: art.count("Dropout") == 2, "Dropout-Schichten vorhanden",
          "Nach jeder versteckten Schicht sollte eine Dropout-Schicht folgen")
kh.pruefe(lambda: baue_fcnn((8,), dropout=0).layers[-2].__class__.__name__ == "Dense",
          "Ohne Dropout wird keine Dropout-Schicht eingefügt", "Bei dropout=0 bitte keine Dropout-Schicht einfügen")

## 4 · Callbacks: Das Training überwachen

| Callback | Aufgabe |
|---|---|
| `EarlyStopping` | beendet das Training, wenn sich `val_loss` `patience` Epochen lang nicht verbessert |
| `ModelCheckpoint` | speichert jeweils die Gewichte der **besten** Epoche – diese laden wir nach dem Training |

In [ ]:
def trainiere(modell, lernrate=LERNRATE, epochen=EPOCHEN, patience=25, name="fcnn", ausgabe=0):
    """Kompiliert, trainiert mit EarlyStopping und lädt die besten Gewichte."""
    modell.compile(optimizer=keras.optimizers.Adam(lernrate),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    pfad = ORDNER / "modelle" / f"{name}_bestes.weights.h5"
    pfad.parent.mkdir(exist_ok=True)
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience),
        keras.callbacks.ModelCheckpoint(pfad, monitor="val_loss", save_best_only=True,
                                        save_weights_only=True),
    ]
    start = time.time()
    verlauf = modell.fit(train_ds, validation_data=val_ds, epochs=epochen,
                         callbacks=callbacks, verbose=ausgabe)
    dauer = time.time() - start
    modell.load_weights(pfad)
    beste = int(np.argmin(verlauf.history["val_loss"]))
    print(f"{len(verlauf.epoch)} Epochen in {dauer:.0f} s · beste Epoche {beste + 1}: "
          f"val_accuracy {verlauf.history['val_accuracy'][beste]:.1%}")
    return verlauf, dauer

In [ ]:
modell = baue_fcnn((256, 128, 64), dropout=0.3)
verlauf, dauer = trainiere(modell, name="fcnn", ausgabe=2)
kh.zeige_lernkurven(verlauf)

## 5 · Lernkurven deuten

### 📖 Interaktiv: Über- und Unteranpassung
Nutze den Simulator und das Diagnose-Training. Deute danach deine eigenen Lernkurven von oben.

In [ ]:
kh.zeige_html("05_lernkurven.html", hoehe=1150)

## 6 · 🔍 Experimente mit Hyperparametern

Die Funktion `experiment()` baut ein Netz, trainiert es und trägt das Ergebnis in eine Tabelle ein. Jede Gruppe untersucht **eine** Fragestellung und präsentiert ihr Ergebnis:

| Gruppe | Fragestellung | Beispielaufrufe |
|---|---|---|
| 1 | Hilft Regularisierung? | `dropout=0` vs. `dropout=0.3` vs. `dropout=0.5`, `aug=False` |
| 2 | Wie groß muss das Netz sein? | `neuronen=(16,)`, `(128, 64)`, `(1024, 512, 256)` |
| 3 | Welche Lernrate? | `lernrate=1e-2`, `1e-3`, `1e-4` |
| 4 | Was bringt mehr Auflösung / Farbe? | in Abschnitt 1 `GROESSE = 64` oder `FARBE = "rgb"` setzen, Daten neu laden |

🍓 Ein Versuch dauert auf dem Raspberry Pi 5 etwa 1 – 3 Minuten.

In [ ]:
ergebnisse = []

def experiment(name, neuronen=(256, 128, 64), dropout=0.3, lernrate=LERNRATE, aug=True):
    global train_ds
    train_ds = kh.als_tf_dataset(X_train, y_train, BATCH, mischen=True,
                                 augmentierung=augmentierung if aug else None)
    m = baue_fcnn(neuronen, dropout)
    v, dauer = trainiere(m, lernrate=lernrate, name="experiment")
    _, test_acc = m.evaluate(test_ds, verbose=0)
    beste = int(np.argmin(v.history["val_loss"]))
    ergebnisse.append({"Versuch": name, "Neuronen": str(neuronen), "Dropout": dropout,
                       "Lernrate": lernrate, "Augm.": aug, "Bild": f"{GROESSE} {FARBE}",
                       "Parameter": m.count_params(), "Epochen": len(v.epoch),
                       "Train-Acc": round(v.history["accuracy"][beste], 3),
                       "Val-Acc": round(v.history["val_accuracy"][beste], 3),
                       "Test-Acc": round(test_acc, 3), "Zeit [s]": round(dauer)})
    kh.zeige_lernkurven(v)
    del m
    return pd.DataFrame(ergebnisse)

In [ ]:
# Beispiel – passe die Aufrufe an eure Fragestellung an:
experiment("ohne Regularisierung", dropout=0, aug=False)
experiment("mit Dropout + Augm.", dropout=0.3, aug=True)

🔍 **Auswertung:** Welche Einstellung war am besten? Woran erkennst du Überanpassung in der Tabelle (Tipp: vergleiche Train-Acc und Val-Acc)? Notiere eure Erkenntnis in einem Satz für die Präsentation.

## 7 · Bewertung des besten Modells

In [ ]:
test_verlust, test_acc = modell.evaluate(test_ds, verbose=0)
print(f"Test-Genauigkeit FCNN: {test_acc:.1%}")

y_pred = modell.predict(X_test, verbose=0).argmax(axis=1)
kh.zeige_konfusionsmatrix(y_test, y_pred, klassen, titel="FCNN – Testdaten")

### ✏️ Aufgabe 2 – Falsch erkannte Bilder anzeigen
Finde die Indizes aller Testbilder, bei denen `y_pred` nicht mit `y_test` übereinstimmt, und zeige sie mit `kh.zeige_beispiele(...)` an. Tipp: `np.where(bedingung)[0]`.

In [ ]:
# >>> LOESUNG
# | falsch = None   # TODO: Indizes der falsch erkannten Testbilder
falsch = np.where(y_pred != y_test)[0]
# <<< LOESUNG

print(f"{len(falsch)} von {len(y_test)} Testbildern falsch erkannt")
kh.pruefe(lambda: len(falsch) == int((y_pred != y_test).sum()), "Alle Fehler gefunden",
          "falsch sollte genau die Indizes enthalten, an denen y_pred != y_test")
if len(falsch):
    kh.zeige_beispiele(X_test[falsch], y_test[falsch], klassen, anzahl=12, vorhersage=y_pred[falsch])

🔍 Was haben die falsch erkannten Bilder gemeinsam (Beleuchtung, Lage, Strichstärke, Bildausschnitt …)? Welche Bilder solltet ihr zusätzlich aufnehmen?

## 8 · Modell speichern
Das Modell wird als `.keras`-Datei, als `.tflite`-Datei (für die Prüfstation) und mit einer Infodatei gespeichert, in der Klassen und Bildgröße stehen.

In [ ]:
kh.speichere_modell(modell, "fcnn", klassen, GROESSE, FARBE,
                    test_genauigkeit=round(float(test_acc), 4), datensatz=DATENSATZ.name)

## 🧪 Teilzielkontrolle 2

In [ ]:
kh.zeige_html("tzk2_neuronale_netze.html", hoehe=900)

### Ausblick
Das FCNN betrachtet jedes Pixel einzeln. Ein Symbol, das ein paar Pixel weiter rechts liegt, erzeugt einen ganz anderen Eingabevektor. **Faltungsnetze (CNN)** in Notebook 12 nutzen dagegen die Nachbarschaft der Pixel – und brauchen dafür sogar weniger Parameter.

In [ ]:
kh.ram_aufraeumen()